# nqs_retrieval — query expansion (NQS): recall delta + save the NQS pool & retrieval features

Qwen synthesizes diagnoses + terms from each patient note (open-model NQS); we re-retrieve BM25+dense with
the expanded query and compare recall to the current pool. If recall improves, we save the NQS pool +
its retrieval features as `pool_nqs.json` / `retrieval_feats_nqs.jsonl`, and re-score the rest of the
pipeline with `pool_tag='nqs'`. NOTE: the expansion is a RETRIEVAL aid — the rerankers keep the original topic.


## Setup (Colab — GPU for Qwen generation + dense encode)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q rank-bm25 sentence-transformers transformers accelerate datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, pickle
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, llm_expand_query, recall_at_k, ndcg_at_k, rrf_fuse
cfg = ExperimentConfig(data_root=DATA_ROOT); CAND_K = cfg.cand_k
SETS = ['trec21','kz','trec22']


In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
sets = load_eval(cfg, SETS)
cur_pool = json.load(open(cfg.path('data/pool_R.json')))
bm25 = pickle.load(open(cfg.bm25_file('fulltext_R'), 'rb'))
doc_emb = np.load(cfg.emb_file('fulltext_R'))
from sentence_transformers import SentenceTransformer
q_enc = SentenceTransformer(cfg.retriever_ckpt); q_enc.max_seq_length = cfg.retriever_max_tokens
topics = {s: [t for t in sets[s]['rel_dict'] if t in sets[s]['topic2text']] for s in SETS}
print('loaded caches; corpus', len(corpus_ids))


In [ ]:
# Diagnosis/term expansions per topic (cached).
EXP_PATH = cfg.path('data/nqs_expansions.jsonl')
from transformers import AutoTokenizer, AutoModelForCausalLM
exp = {}
if os.path.exists(EXP_PATH):
    for l in open(EXP_PATH):
        r = json.loads(l); exp[(r['source'], r['topic_id'])] = r['expansion']
else:
    tok = AutoTokenizer.from_pretrained(cfg.llm_ckpt)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    llm = AutoModelForCausalLM.from_pretrained(cfg.llm_ckpt, torch_dtype=torch.float16, device_map='auto').eval()
    with open(EXP_PATH, 'w') as out:
        for s in SETS:
            for t in tqdm(topics[s], desc=f'expand {s}'):
                e = llm_expand_query(llm, tok, sets[s]['topic2text'][t], cfg)
                exp[(s,t)] = e; out.write(json.dumps({'source':s,'topic_id':t,'expansion':e})+'\n')
    del llm; torch.cuda.empty_cache()


In [ ]:
# Retrieve with the EXPANDED query; capture per-doc bm25/dense/rrf features (like eval_fullcorpus).
def retrieve(qtext, k=CAND_K):
    sc = bm25.get_scores(qtext.lower().split()); bt = np.argpartition(-sc,k)[:k]; bt=bt[np.argsort(-sc[bt])]
    qv = q_enc.encode([qtext], normalize_embeddings=True)[0].astype('float32'); sims = doc_emb@qv
    dt = np.argpartition(-sims,k)[:k]; dt=dt[np.argsort(-sims[dt])]
    bm = {corpus_ids[i]: float(sc[i]) for i in bt}; dn = {corpus_ids[i]: float(sims[i]) for i in dt}
    br = {d:r for r,d in enumerate(bm)}; dr = {d:r for r,d in enumerate(dn)}
    rrf = rrf_fuse([list(br), list(dr)], k=cfg.rrf_k)
    cand = sorted(set(bm)|set(dn), key=lambda d: rrf.get(d,0), reverse=True)
    feats = {d: {'bm25':bm.get(d,0.), 'dense':dn.get(d,0.), 'rrf':rrf.get(d,0.),
                 'bm25_rank':br.get(d,k), 'dense_rank':dr.get(d,k)} for d in cand}
    return cand, feats
nqs_pool, nqs_feats = {s:{} for s in SETS}, {}
for s in SETS:
    for t in tqdm(topics[s], desc=f'nqs retrieve {s}'):
        cand, feats = retrieve(sets[s]['topic2text'][t] + ' ' + exp[(s,t)])
        nqs_pool[s][t] = cand
        for d in cand: nqs_feats[(s,t,d)] = feats[d]


In [ ]:
# Recall@1000 + oracle: current vs NQS vs union (union measured over the FULL combined pool, not @1000).
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']
    def R(getp, k=1000): return np.mean([recall_at_k(getp(t), rel[t], k, rel_level=2) for t in topics[s]])
    def O(getp): return np.mean([ndcg_at_k(sorted(getp(t), key=lambda d: rel[t].get(d,0), reverse=True), rel[t]) for t in topics[s]])
    union = {t: list(dict.fromkeys(cur_pool[s][t] + nqs_pool[s][t])) for t in topics[s]}
    rows.append({'split':s, 'recall_cur':round(float(R(lambda t: cur_pool[s][t])),3),
                 'recall_nqs':round(float(R(lambda t: nqs_pool[s][t])),3),
                 'recall_union':round(float(np.mean([recall_at_k(union[t], rel[t], len(union[t]), rel_level=2) for t in topics[s]])),3),
                 'oracle_nqs':round(float(O(lambda t: nqs_pool[s][t])),3)})
pd.DataFrame(rows)


In [ ]:
# Where NQS helps most on trec22 (per-topic recall gain, NQS vs current).
rel = sets['trec22']['rel_dict']; g=[]
for t in topics['trec22']:
    rc = recall_at_k(cur_pool['trec22'][t], rel[t], 1000, rel_level=2)
    rn = recall_at_k(nqs_pool['trec22'][t], rel[t], 1000, rel_level=2)
    g.append({'topic':t, 'recall_cur':round(rc,3), 'recall_nqs':round(rn,3), 'gain':round(rn-rc,3),
              'topic_text':sets['trec22']['topic2text'][t].strip()[:70]})
pd.DataFrame(sorted(g, key=lambda x:-x['gain'])[:10])


In [ ]:
# Save the NQS pool + its retrieval features (pool_tag='nqs') for the downstream re-score.
json.dump(nqs_pool, open(cfg.path('data/pool_nqs.json'), 'w'))
with open(cfg.path('data/retrieval_feats_nqs.jsonl'), 'w') as f:
    for (s,t,d), v in nqs_feats.items():
        f.write(json.dumps({'source':s, 'topic_id':t, 'doc_id':d, **v}) + '\n')
print('wrote data/pool_nqs.json + data/retrieval_feats_nqs.jsonl (pool_tag=nqs)')


## Reading it
- `recall_nqs` (0.658 on trec22 last run) vs `recall_cur` (0.543) is the real signal — NQS retrieval is
  the go/no-go. `recall_union` (now correct, full-pool) shows the ceiling if you kept both.
- To convert this to NDCG: re-run `rerank_llm_feature`, `rerank_condition_match`, `train_ensemble_full`
  each with `POOL_TAG='nqs'` (top of the notebook) — they'll score the NQS pool via cfg.pool_path()/feat_file().
- Watch per-topic NDCG on the implicit-diagnosis topics — retrieval is a tail lever.
